# Complete Short Put Research Workflow

This notebook demonstrates the end-to-end workflow for the options research framework. It uses a small synthetic SPY dataset because historical data ingestion is not wired up yet. When the data pipeline is available, replace the synthetic data section with calls to the ThetaData provider and DuckDB repositories.

Workflow covered:

1. Load historical data.
2. Select contracts.
3. Run the short put strategy.
4. Execute the backtest.
5. Calculate analytics.
6. Generate reports.


## 0. Setup

The notebook assumes it is run from either the repository root or the notebooks directory. The local src path is added for editable-project usage.


In [ ]:
# ruff: noqa: E402, I001
from __future__ import annotations

from datetime import UTC, date, datetime
from decimal import Decimal
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from options_quant.analytics import PerformanceAnalyzer
from options_quant.backtest import (
    BacktestConfig,
    BacktestEngine,
    BacktestMarketEvent,
    BacktestResult,
)
from options_quant.data.models import (
    OptionChain,
    OptionContract,
    OptionGreek,
    OptionQuote,
    OptionType,
    UnderlyingPrice,
)
from options_quant.reporting import BacktestReportGenerator
from options_quant.strategies import (
    ContractSelectionEngine,
    OptionSelectionQuery,
    PortfolioState,
    ShortPutStrategy,
    ShortPutStrategyConfig,
    StrategyMarketData,
)


## 1. Load Historical Data

In production this section should load data from the configured provider/storage layer, for example ThetaData into DuckDB repositories. For now, it creates synthetic SPY market data that mirrors the typed models used by the rest of the framework.


In [ ]:
as_of = date(2026, 1, 2)
next_day = date(2026, 1, 3)
observed_at = datetime(2026, 1, 2, 14, 30, tzinfo=UTC)

spy_underlying = UnderlyingPrice(
    symbol="SPY",
    timestamp=observed_at,
    price=Decimal("520"),
)

spy_put_30_dte = OptionContract(
    underlying_symbol="SPY",
    expiration=date(2026, 2, 1),
    strike=Decimal("505"),
    option_type=OptionType.PUT,
)
spy_put_38_dte = OptionContract(
    underlying_symbol="SPY",
    expiration=date(2026, 2, 9),
    strike=Decimal("500"),
    option_type=OptionType.PUT,
)
spy_put_45_dte = OptionContract(
    underlying_symbol="SPY",
    expiration=date(2026, 2, 16),
    strike=Decimal("495"),
    option_type=OptionType.PUT,
)

chain = OptionChain(
    underlying_symbol="SPY",
    timestamp=observed_at,
    contracts=(spy_put_30_dte, spy_put_38_dte, spy_put_45_dte),
)

option_greeks = {
    spy_put_30_dte: OptionGreek(
        contract=spy_put_30_dte,
        timestamp=observed_at,
        delta=Decimal("-0.18"),
        implied_volatility=Decimal("0.22"),
    ),
    spy_put_38_dte: OptionGreek(
        contract=spy_put_38_dte,
        timestamp=observed_at,
        delta=Decimal("-0.11"),
        implied_volatility=Decimal("0.24"),
    ),
    spy_put_45_dte: OptionGreek(
        contract=spy_put_45_dte,
        timestamp=observed_at,
        delta=Decimal("-0.09"),
        implied_volatility=Decimal("0.25"),
    ),
}

entry_quotes = {
    spy_put_30_dte: OptionQuote(
        contract=spy_put_30_dte,
        timestamp=observed_at,
        bid=Decimal("1.40"),
        ask=Decimal("1.50"),
        mark=Decimal("1.45"),
    ),
    spy_put_38_dte: OptionQuote(
        contract=spy_put_38_dte,
        timestamp=observed_at,
        bid=Decimal("1.90"),
        ask=Decimal("2.10"),
        mark=Decimal("2.00"),
    ),
    spy_put_45_dte: OptionQuote(
        contract=spy_put_45_dte,
        timestamp=observed_at,
        bid=Decimal("2.20"),
        ask=Decimal("2.40"),
        mark=Decimal("2.30"),
    ),
}

print(f"Loaded {len(chain.contracts)} synthetic SPY contracts.")


## 2. Select Contracts

Use the contract selection engine to find the 30-45 DTE SPY put closest to the strategy target delta of -0.10.


In [ ]:
selection_engine = ContractSelectionEngine(
    chain,
    spy_underlying.price,
    as_of_date=as_of,
    greeks=list(option_greeks.values()),
)

selected_candidate = selection_engine.best(
    OptionSelectionQuery(
        option_type=OptionType.PUT,
        min_dte=30,
        max_dte=45,
        target_delta=Decimal("-0.10"),
    )
)

selected_candidate


## 3. Run the Short Put Strategy

The strategy receives typed market data and portfolio state. It emits a strongly typed signal, then converts that signal into an order event for the backtest engine.


In [ ]:
strategy = ShortPutStrategy(
    ShortPutStrategyConfig(
        position_size=1,
        take_profit_pct=Decimal("0.50"),
        stop_loss_pct=Decimal("1.00"),
    )
)

entry_market_data = StrategyMarketData(
    date=as_of,
    underlying_prices={"SPY": spy_underlying},
    option_chains={"SPY": chain},
    option_quotes=entry_quotes,
    option_greeks=option_greeks,
)

empty_portfolio = PortfolioState(
    date=as_of,
    cash_balance=Decimal("100000"),
    realized_pnl=Decimal("0"),
    unrealized_pnl=Decimal("0"),
    capital_utilization=Decimal("0"),
    equity=Decimal("100000"),
    open_positions=(),
)

signals = strategy.generate_signals(entry_market_data, empty_portfolio)
entry_orders = strategy.manage_positions(entry_market_data, empty_portfolio, signals)

signals, entry_orders


## 4. Execute the Backtest

This example opens the selected short put on day one, then marks the option lower on day two so the strategy take-profit rule emits a closing order. In a full historical run, a driver loop would repeat this daily over provider/repository data.


In [ ]:
engine = BacktestEngine(
    BacktestConfig(
        initial_cash=Decimal("100000"),
        commission_per_contract=Decimal("0.65"),
        slippage_per_contract=Decimal("0.05"),
    )
)

entry_result = engine.run(
    [
        BacktestMarketEvent(
            date=as_of,
            option_marks={spy_put_38_dte: Decimal("2.00")},
        )
    ],
    {as_of: list(entry_orders)},
)

open_position = entry_result.snapshots[-1].open_positions[0]

exit_quote = OptionQuote(
    contract=open_position.contract,
    timestamp=datetime(2026, 1, 3, 14, 30, tzinfo=UTC),
    bid=Decimal("0.85"),
    ask=Decimal("0.95"),
    mark=Decimal("0.90"),
)
exit_market_data = entry_market_data.model_copy(
    update={
        "date": next_day,
        "option_quotes": {open_position.contract: exit_quote},
    }
)
portfolio_after_entry = PortfolioState.from_account_snapshot(entry_result.snapshots[-1])
exit_orders = strategy.exit_rules(exit_market_data, portfolio_after_entry)

exit_result = engine.run(
    [
        BacktestMarketEvent(
            date=next_day,
            option_marks={open_position.contract: Decimal("0.90")},
        )
    ],
    {next_day: list(exit_orders)},
)

backtest_result = BacktestResult(
    snapshots=entry_result.snapshots + exit_result.snapshots,
    closed_positions=exit_result.closed_positions,
)

backtest_result


## 5. Calculate Analytics

The analytics module consumes the completed BacktestResult. Benchmark returns can be supplied later for alpha, beta, tracking error, and information ratio.


In [ ]:
analyzer = PerformanceAnalyzer()
performance_report = analyzer.analyze(backtest_result)

performance_report


## 6. Generate Reports

The reporting API can produce summary statistics, a trade log CSV, an equity curve chart, and a drawdown chart. Outputs are written under notebooks/generated_reports, which is ignored by git.


In [ ]:
report_generator = BacktestReportGenerator()
output_dir = repo_root / "notebooks" / "generated_reports" / "short_put_demo"
artifacts = report_generator.generate_report(backtest_result, output_dir)

summary_table = [(row.name, row.value) for row in artifacts.summary_statistics]
summary_table[:8], artifacts


## Next Steps

- Replace the synthetic data cell with ThetaData provider calls and DuckDB repository reads.
- Wrap the strategy and backtest calls in a reusable daily simulation loop.
- Add benchmark returns to the analytics cell once benchmark data is available.
- Expand reporting outputs as strategy requirements become clearer.
